# Installación de librerías

toca correrlo dos veces, la primera falla xd

In [ ]:
!pip install tensorflow==2.12.0
!pip install mtcnn
!pip install dlib imutils

In [ ]:
from google.colab import drive
import os
from PIL import Image
import cv2
import shutil
import torch
from IPython.display import display
from mtcnn import MTCNN
from mtcnn.utils.images import load_image
import matplotlib.pyplot as plt
import dlib
import numpy as np
from imutils.face_utils import FaceAligner
from imutils.face_utils import rect_to_bb

In [ ]:
# Acá se montan todos los archivos del drive, es necesario que le den las credenciales de la U
drive.mount('/content/drive')


# Definición de funciónes y filtros


Reducción de ruido

In [ ]:
def reducir_ruido(imagen):
    imagen_procesada = cv2.fastNlMeansDenoisingColored(
      imagen,
      None,
      h=10,         # fuerza del filtro para luminancia (mayor = más suavizado)
      hColor=10,    # fuerza del filtro para los canales de color
      templateWindowSize=7,
      searchWindowSize=21
      )
    print("Imagen despues de reducción de ruido")
    plt.imshow(cv2.cvtColor(imagen_procesada, cv2.COLOR_BGR2RGB))
    plt.show()
    return imagen_procesada

Correción de iluminación

In [ ]:
def corregir_iluminacion(imagen):

    # Convertir a espacio de color LAB (Luminancia, A, B)
    lab = cv2.cvtColor(imagen, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # Crear objeto CLAHE (limita el contraste para evitar artefactos)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    # Aplicar CLAHE solo al canal de luminancia (L)
    l_ecualizado = clahe.apply(l)

    # Recomponer la imagen LAB con la luminancia corregida
    lab_corregida = cv2.merge((l_ecualizado, a, b))

    # Convertir de nuevo a BGR
    imagen_corregida = cv2.cvtColor(lab_corregida, cv2.COLOR_LAB2BGR)
    print("Imagen despues de corregir ilumacion")
    plt.imshow(cv2.cvtColor(imagen_corregida, cv2.COLOR_BGR2RGB))
    plt.show()
    return imagen_corregida


Recorte de cara

In [ ]:
from mtcnn import MTCNN
import cv2
import matplotlib.pyplot as plt

detector_mtcnn = MTCNN()

def recortar_cara(imagen):
    # Convertir la imagen a RGB para MTCNN
    imagen_corregida = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)

    # Detección de rostros
    resultados = detector_mtcnn.detect_faces(imagen_corregida)

    if len(resultados) == 0:
        print("No se detectó ningún rostro en la imagen.")
        plt.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
        plt.show()
        return imagen, None, None  # Devolver imagen original y None para confidence y keypoints

    # Obtener el primer rostro detectado
    result = resultados[0]
    x, y, w, h = result['box']
    confidence = result['confidence']
    keypoints = result['keypoints']

    # Recortar la imagen
    imagen_recortada = imagen[y:y+h, x:x+w]
    print("Imagen después de recorte")
    plt.imshow(cv2.cvtColor(imagen_recortada, cv2.COLOR_BGR2RGB))
    plt.show()

    return imagen_recortada, confidence, keypoints

Alineación facial y resize

In [ ]:
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2

In [ ]:
SHAPE_PREDICTOR_PATH = "shape_predictor_68_face_landmarks.dat"
OUTPUT_SIZE = (224, 224)

# Inicializar el detector de rostros y el predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(SHAPE_PREDICTOR_PATH)

In [ ]:
def align_and_resize_face(image: np.ndarray) -> np.ndarray:
    if image is None or not isinstance(image, np.ndarray):
        print("Imagen inválida. Devolviendo imagen negra redimensionada.")
        return cv2.resize(np.zeros((100, 100, 3), dtype=np.uint8), OUTPUT_SIZE)

    original_image = image.copy()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = detector(gray, 1)

    if len(faces) == 0:
        print("No se detectó ningún rostro. Redimensionando imagen original.")
        return cv2.resize(original_image, OUTPUT_SIZE)

    face = faces[0]
    landmarks = predictor(gray, face)
    landmarks_np = np.array([[p.x, p.y] for p in landmarks.parts()], dtype=np.float32)

    left_eye = np.mean(landmarks_np[36:42], axis=0)
    right_eye = np.mean(landmarks_np[42:48], axis=0)

    # Calcular ángulo
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    angle = np.degrees(np.arctan2(dy, dx))

    # Calcular centro entre los ojos
    eyes_center = ((left_eye[0] + right_eye[0]) / 2,
                   (left_eye[1] + right_eye[1]) / 2)

    # Distancia entre ojos en la imagen original
    dist = np.sqrt(dx**2 + dy**2)

    # Distancia deseada entre ojos
    desired_eye_dist = OUTPUT_SIZE[0] * (0.68 - 0.32)
    scale = desired_eye_dist / dist

    # Obtener matriz de rotación + escala
    M = cv2.getRotationMatrix2D(eyes_center, angle, scale)

    # Mover los ojos a la posición deseada
    tx = OUTPUT_SIZE[0] * 0.5 - eyes_center[0]
    ty = OUTPUT_SIZE[1] * 0.4 - eyes_center[1]
    M[0, 2] += tx
    M[1, 2] += ty

    aligned = cv2.warpAffine(image, M, OUTPUT_SIZE, flags=cv2.INTER_LINEAR)

    print("Imagen alineada sin distorsión:")
    plt.imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

    return aligned

Normalización de las imagenes


In [ ]:
# pasa todos los valores RGB a un
def normalizacion(imagen):
    imagen_procesada = imagen.astype('float32')/255.0
    return imagen_procesada

In [ ]:
# Función principal para aplicar todo el pipeline
# (acá se llaman las funciones de cada filtro)
import cv2
import matplotlib.pyplot as plt

def procesar_imagen(imagen):
    print("Imagen original")
    plt.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
    plt.show()

    # Aplicar el pipeline
    imagen_procesada = reducir_ruido(imagen)
    imagen_procesada = corregir_iluminacion(imagen_procesada)
    imagen_procesada, confidence, keypoints = recortar_cara(imagen_procesada)
    imagen_procesada = align_and_resize_face(imagen_procesada)
    imagen_procesada = normalizacion(imagen_procesada)

    return imagen_procesada, confidence, keypoints

# Aplicación de filtros a datasets de prueba


In [ ]:
# Rutas a las carpetas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Imagenes(prueba)')
output_folder = os.path.join(shared_drive_path, 'Dataset(prueba)')
print(os.listdir(input_folder))

for filename in os.listdir(input_folder):

    input_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    # Load an image
    image = cv2.imread(input_path)
    imagen_procesada = procesar_imagen(image)
    cv2.imwrite(output_path, imagen_procesada)

#Generar lotes

In [ ]:

"""
import os
import shutil
import random

# Rutas a las carpetas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Imagenes')
output_base_folder = os.path.join(shared_drive_path, 'Dataset')

# Nombres de las personas
personas = ['gabriela', 'david', 'manuel', 'huertas']

# Crear carpeta principal Dataset si no existe
if not os.path.exists(output_base_folder):
    os.makedirs(output_base_folder)

# Obtener lista de todas las imágenes
image_files = [f for f in os.listdir(input_folder) if os.path.isfile(os.path.join(input_folder, f))]
random.shuffle(image_files)  # Mezclar aleatoriamente las imágenes

# Calcular cuántas imágenes por persona (1200 por persona = 2 lotes de 600)
images_per_person = 1200
images_per_batch = 600

# Dividir las imágenes
for i, persona in enumerate(personas):
    # Crear carpeta para cada persona
    persona_folder = os.path.join(output_base_folder, f'dataset-{persona}')
    if not os.path.exists(persona_folder):
        os.makedirs(persona_folder)

    # Crear dos lotes
    for batch in range(1, 3):  # Lote 1 y Lote 2
        batch_folder = os.path.join(persona_folder, f'lote-{batch}')
        if not os.path.exists(batch_folder):
            os.makedirs(batch_folder)

        # Seleccionar imágenes para este lote
        start_idx = (i * images_per_person) + ((batch - 1) * images_per_batch)
        end_idx = start_idx + images_per_batch
        batch_images = image_files[start_idx:end_idx]

        # Copiar imágenes al lote correspondiente
        for img in batch_images:
            src_path = os.path.join(input_folder, img)
            dst_path = os.path.join(batch_folder, img)
            shutil.copy2(src_path, dst_path)  # Copia la imagen manteniendo metadatos

    print(f"Carpeta dataset-{persona} creada con 2 lotes de {images_per_batch} imágenes cada uno.")

print("División del dataset completada.")
"""

# Aplicación de filtros a dataset completo


# Lotes Manuel

## LOTE 1

In [ ]:
import os
import cv2
import csv
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Dataset', 'dataset-manuel', 'lote-2')
output_folder = os.path.join(shared_drive_path, 'Dataset', 'preprocesadas')
csv_path = os.path.join(shared_drive_path, 'Dataset', 'processed_images.csv')

# Crear carpeta de salida si no existe
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Definir encabezados del CSV
csv_headers = ['filename', 'output_path', 'sexo', 'edad', 'raza', 'mtcnn_confidence',
               'nose_x', 'nose_y', 'mouth_right_x', 'mouth_right_y',
               'right_eye_x', 'right_eye_y', 'left_eye_x', 'left_eye_y',
               'mouth_left_x', 'mouth_left_y']

# Crear CSV si no existe
if not os.path.exists(csv_path):
    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(csv_headers)

# Diccionarios para mapear valores numéricos de sexo y raza
sexo_map = {
    '0': 'Hombre',
    '1': 'Mujer'
}

raza_map = {
    '0': 'Blanco',
    '1': 'Negro',
    '2': 'Asiático',
    '3': 'Indio',
    '4': 'Otros'
}

# Procesar imágenes
for filename in os.listdir(input_folder):
    input_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    # Verificar que el archivo es una imagen
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        try:
            image = cv2.imread(input_path)
            if image is None:
                print(f"Error: No se pudo cargar la imagen {filename}")
                continue
            imagen_procesada, confidence, keypoints = procesar_imagen(image)
            # Convertir de vuelta a formato uint8 para guardar
            imagen_procesada = (imagen_procesada * 255).astype('uint8')
            cv2.imwrite(output_path, imagen_procesada)
            print(f"Imagen procesada: {filename}")

            # Extraer sexo, edad y raza del nombre del archivo
            try:
                parts = filename.split('_')
                print(f"Procesando archivo: {filename}, partes: {parts}")  # Depuración
                if len(parts) < 4:
                    raise ValueError("Formato de nombre de archivo inválido")

                edad = parts[0] if parts[0].isdigit() else "desconocido"
                sexo = sexo_map.get(parts[1], "desconocido") if parts[1] in sexo_map else "desconocido"
                raza = raza_map.get(parts[2], "desconocido") if parts[2] in raza_map else "desconocido"
                print(f"Extraído - Edad: {edad}, Sexo: {sexo}, Raza: {raza}")  # Depuración

            except Exception as parse_error:
                print(f"Error al parsear el nombre del archivo {filename}: {str(parse_error)}")
                edad = "desconocido"
                sexo = "desconocido"
                raza = "desconocido"

            # Preparar datos para el CSV
            csv_row = [
                filename,
                output_path,
                sexo,
                edad,
                raza,
                confidence if confidence is not None else "",
                keypoints['nose'][0] if keypoints and 'nose' in keypoints else "",
                keypoints['nose'][1] if keypoints and 'nose' in keypoints else "",
                keypoints['mouth_right'][0] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['mouth_right'][1] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['right_eye'][0] if keypoints and 'right_eye' in keypoints else "",
                keypoints['right_eye'][1] if keypoints and 'right_eye' in keypoints else "",
                keypoints['left_eye'][0] if keypoints and 'left_eye' in keypoints else "",
                keypoints['left_eye'][1] if keypoints and 'left_eye' in keypoints else "",
                keypoints['mouth_left'][0] if keypoints and 'mouth_left' in keypoints else "",
                keypoints['mouth_left'][1] if keypoints and 'mouth_left' in keypoints else ""
            ]

            # Escribir en el CSV
            with open(csv_path, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(csv_row)

        except Exception as e:
            print(f"Error al procesar {filename}: {str(e)}")
    else:
        print(f"Archivo ignorado (no es imagen): {filename}")

## LOTE 2

In [ ]:
import os
import cv2
import csv
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Dataset', 'dataset-manuel', 'lote-2')
output_folder = os.path.join(shared_drive_path, 'Dataset', 'preprocesadas')
csv_path = os.path.join(shared_drive_path, 'Dataset', 'processed_images.csv')

# Crear carpeta de salida si no existe
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Definir encabezados del CSV
csv_headers = ['filename', 'output_path', 'sexo', 'edad', 'raza', 'mtcnn_confidence',
               'nose_x', 'nose_y', 'mouth_right_x', 'mouth_right_y',
               'right_eye_x', 'right_eye_y', 'left_eye_x', 'left_eye_y',
               'mouth_left_x', 'mouth_left_y']

# Crear CSV si no existe
if not os.path.exists(csv_path):
    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(csv_headers)

# Diccionarios para mapear valores numéricos de sexo y raza
sexo_map = {
    '0': 'Hombre',
    '1': 'Mujer'
}

raza_map = {
    '0': 'Blanco',
    '1': 'Negro',
    '2': 'Asiático',
    '3': 'Indio',
    '4': 'Otros'
}

# Procesar imágenes
for filename in os.listdir(input_folder):
    input_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    # Verificar que el archivo es una imagen
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        try:
            image = cv2.imread(input_path)
            if image is None:
                print(f"Error: No se pudo cargar la imagen {filename}")
                continue
            imagen_procesada, confidence, keypoints = procesar_imagen(image)
            # Convertir de vuelta a formato uint8 para guardar
            imagen_procesada = (imagen_procesada * 255).astype('uint8')
            cv2.imwrite(output_path, imagen_procesada)
            print(f"Imagen procesada: {filename}")

            # Extraer sexo, edad y raza del nombre del archivo
            try:
                parts = filename.split('_')
                print(f"Procesando archivo: {filename}, partes: {parts}")  # Depuración
                if len(parts) < 4:
                    raise ValueError("Formato de nombre de archivo inválido")

                edad = parts[0] if parts[0].isdigit() else "desconocido"
                sexo = sexo_map.get(parts[1], "desconocido") if parts[1] in sexo_map else "desconocido"
                raza = raza_map.get(parts[2], "desconocido") if parts[2] in raza_map else "desconocido"
                print(f"Extraído - Edad: {edad}, Sexo: {sexo}, Raza: {raza}")  # Depuración

            except Exception as parse_error:
                print(f"Error al parsear el nombre del archivo {filename}: {str(parse_error)}")
                edad = "desconocido"
                sexo = "desconocido"
                raza = "desconocido"

            # Preparar datos para el CSV
            csv_row = [
                filename,
                output_path,
                sexo,
                edad,
                raza,
                confidence if confidence is not None else "",
                keypoints['nose'][0] if keypoints and 'nose' in keypoints else "",
                keypoints['nose'][1] if keypoints and 'nose' in keypoints else "",
                keypoints['mouth_right'][0] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['mouth_right'][1] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['right_eye'][0] if keypoints and 'right_eye' in keypoints else "",
                keypoints['right_eye'][1] if keypoints and 'right_eye' in keypoints else "",
                keypoints['left_eye'][0] if keypoints and 'left_eye' in keypoints else "",
                keypoints['left_eye'][1] if keypoints and 'left_eye' in keypoints else "",
                keypoints['mouth_left'][0] if keypoints and 'mouth_left' in keypoints else "",
                keypoints['mouth_left'][1] if keypoints and 'mouth_left' in keypoints else ""
            ]

            # Escribir en el CSV
            with open(csv_path, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(csv_row)

        except Exception as e:
            print(f"Error al procesar {filename}: {str(e)}")
    else:
        print(f"Archivo ignorado (no es imagen): {filename}")

# Lotes David

## LOTE 1

In [ ]:
import os
import cv2
import csv
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Dataset', 'dataset-david', 'lote-1')
output_folder = os.path.join(shared_drive_path, 'Dataset', 'preprocesadas')
csv_path = os.path.join(shared_drive_path, 'Dataset', 'processed_images.csv')

# Crear carpeta de salida si no existe
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Definir encabezados del CSV
csv_headers = ['filename', 'output_path', 'sexo', 'edad', 'raza', 'mtcnn_confidence',
               'nose_x', 'nose_y', 'mouth_right_x', 'mouth_right_y',
               'right_eye_x', 'right_eye_y', 'left_eye_x', 'left_eye_y',
               'mouth_left_x', 'mouth_left_y']

# Crear CSV si no existe
if not os.path.exists(csv_path):
    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(csv_headers)

# Diccionarios para mapear valores numéricos de sexo y raza
sexo_map = {
    '0': 'Hombre',
    '1': 'Mujer'
}

raza_map = {
    '0': 'Blanco',
    '1': 'Negro',
    '2': 'Asiático',
    '3': 'Indio',
    '4': 'Otros'
}

# Procesar imágenes
for filename in os.listdir(input_folder):
    input_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    # Verificar que el archivo es una imagen
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        try:
            image = cv2.imread(input_path)
            if image is None:
                print(f"Error: No se pudo cargar la imagen {filename}")
                continue
            imagen_procesada, confidence, keypoints = procesar_imagen(image)
            # Convertir de vuelta a formato uint8 para guardar
            imagen_procesada = (imagen_procesada * 255).astype('uint8')
            cv2.imwrite(output_path, imagen_procesada)
            print(f"Imagen procesada: {filename}")

            # Extraer sexo, edad y raza del nombre del archivo
            try:
                parts = filename.split('_')
                print(f"Procesando archivo: {filename}, partes: {parts}")  # Depuración
                if len(parts) < 4:
                    raise ValueError("Formato de nombre de archivo inválido")

                edad = parts[0] if parts[0].isdigit() else "desconocido"
                sexo = sexo_map.get(parts[1], "desconocido") if parts[1] in sexo_map else "desconocido"
                raza = raza_map.get(parts[2], "desconocido") if parts[2] in raza_map else "desconocido"
                print(f"Extraído - Edad: {edad}, Sexo: {sexo}, Raza: {raza}")  # Depuración

            except Exception as parse_error:
                print(f"Error al parsear el nombre del archivo {filename}: {str(parse_error)}")
                edad = "desconocido"
                sexo = "desconocido"
                raza = "desconocido"

            # Preparar datos para el CSV
            csv_row = [
                filename,
                output_path,
                sexo,
                edad,
                raza,
                confidence if confidence is not None else "",
                keypoints['nose'][0] if keypoints and 'nose' in keypoints else "",
                keypoints['nose'][1] if keypoints and 'nose' in keypoints else "",
                keypoints['mouth_right'][0] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['mouth_right'][1] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['right_eye'][0] if keypoints and 'right_eye' in keypoints else "",
                keypoints['right_eye'][1] if keypoints and 'right_eye' in keypoints else "",
                keypoints['left_eye'][0] if keypoints and 'left_eye' in keypoints else "",
                keypoints['left_eye'][1] if keypoints and 'left_eye' in keypoints else "",
                keypoints['mouth_left'][0] if keypoints and 'mouth_left' in keypoints else "",
                keypoints['mouth_left'][1] if keypoints and 'mouth_left' in keypoints else ""
            ]

            # Escribir en el CSV
            with open(csv_path, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(csv_row)

        except Exception as e:
            print(f"Error al procesar {filename}: {str(e)}")
    else:
        print(f"Archivo ignorado (no es imagen): {filename}")

## LOTE 2

In [ ]:
import os
import cv2
import csv
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Dataset', 'dataset-david', 'lote-2')
output_folder = os.path.join(shared_drive_path, 'Dataset', 'preprocesadas')
csv_path = os.path.join(shared_drive_path, 'Dataset', 'processed_images.csv')

# Crear carpeta de salida si no existe
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Definir encabezados del CSV
csv_headers = ['filename', 'output_path', 'sexo', 'edad', 'raza', 'mtcnn_confidence',
               'nose_x', 'nose_y', 'mouth_right_x', 'mouth_right_y',
               'right_eye_x', 'right_eye_y', 'left_eye_x', 'left_eye_y',
               'mouth_left_x', 'mouth_left_y']

# Crear CSV si no existe
if not os.path.exists(csv_path):
    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(csv_headers)

# Diccionarios para mapear valores numéricos de sexo y raza
sexo_map = {
    '0': 'Hombre',
    '1': 'Mujer'
}

raza_map = {
    '0': 'Blanco',
    '1': 'Negro',
    '2': 'Asiático',
    '3': 'Indio',
    '4': 'Otros'
}

# Procesar imágenes
for filename in os.listdir(input_folder):
    input_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    # Verificar que el archivo es una imagen
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        try:
            image = cv2.imread(input_path)
            if image is None:
                print(f"Error: No se pudo cargar la imagen {filename}")
                continue
            imagen_procesada, confidence, keypoints = procesar_imagen(image)
            # Convertir de vuelta a formato uint8 para guardar
            imagen_procesada = (imagen_procesada * 255).astype('uint8')
            cv2.imwrite(output_path, imagen_procesada)
            print(f"Imagen procesada: {filename}")

            # Extraer sexo, edad y raza del nombre del archivo
            try:
                parts = filename.split('_')
                print(f"Procesando archivo: {filename}, partes: {parts}")  # Depuración
                if len(parts) < 4:
                    raise ValueError("Formato de nombre de archivo inválido")

                edad = parts[0] if parts[0].isdigit() else "desconocido"
                sexo = sexo_map.get(parts[1], "desconocido") if parts[1] in sexo_map else "desconocido"
                raza = raza_map.get(parts[2], "desconocido") if parts[2] in raza_map else "desconocido"
                print(f"Extraído - Edad: {edad}, Sexo: {sexo}, Raza: {raza}")  # Depuración

            except Exception as parse_error:
                print(f"Error al parsear el nombre del archivo {filename}: {str(parse_error)}")
                edad = "desconocido"
                sexo = "desconocido"
                raza = "desconocido"

            # Preparar datos para el CSV
            csv_row = [
                filename,
                output_path,
                sexo,
                edad,
                raza,
                confidence if confidence is not None else "",
                keypoints['nose'][0] if keypoints and 'nose' in keypoints else "",
                keypoints['nose'][1] if keypoints and 'nose' in keypoints else "",
                keypoints['mouth_right'][0] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['mouth_right'][1] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['right_eye'][0] if keypoints and 'right_eye' in keypoints else "",
                keypoints['right_eye'][1] if keypoints and 'right_eye' in keypoints else "",
                keypoints['left_eye'][0] if keypoints and 'left_eye' in keypoints else "",
                keypoints['left_eye'][1] if keypoints and 'left_eye' in keypoints else "",
                keypoints['mouth_left'][0] if keypoints and 'mouth_left' in keypoints else "",
                keypoints['mouth_left'][1] if keypoints and 'mouth_left' in keypoints else ""
            ]

            # Escribir en el CSV
            with open(csv_path, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(csv_row)

        except Exception as e:
            print(f"Error al procesar {filename}: {str(e)}")
    else:
        print(f"Archivo ignorado (no es imagen): {filename}")

# Lotes Huertas

## LOTE 1

In [ ]:
import os
import cv2
import csv
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Dataset')
output_folder = os.path.join(shared_drive_path, 'Dataset Preprocesado')

# Crear carpeta de salida si no existe
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Definir encabezados del CSV
csv_headers = ['filename', 'output_path', 'sexo', 'edad', 'raza', 'mtcnn_confidence',
               'nose_x', 'nose_y', 'mouth_right_x', 'mouth_right_y',
               'right_eye_x', 'right_eye_y', 'left_eye_x', 'left_eye_y',
               'mouth_left_x', 'mouth_left_y']

# Crear CSV si no existe
if not os.path.exists(csv_path):
    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(csv_headers)

# Diccionarios para mapear valores numéricos de sexo y raza
sexo_map = {
    '0': 'Hombre',
    '1': 'Mujer'
}

raza_map = {
    '0': 'Blanco',
    '1': 'Negro',
    '2': 'Asiático',
    '3': 'Indio',
    '4': 'Otros'
}

# Procesar imágenes
for filename in os.listdir(input_folder):
    input_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    # Verificar que el archivo es una imagen
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        try:
            image = cv2.imread(input_path)
            if image is None:
                print(f"Error: No se pudo cargar la imagen {filename}")
                continue
            imagen_procesada, confidence, keypoints = procesar_imagen(image)
            # Convertir de vuelta a formato uint8 para guardar
            imagen_procesada = (imagen_procesada * 255).astype('uint8')
            cv2.imwrite(output_path, imagen_procesada)
            print(f"Imagen procesada: {filename}")

            # Extraer sexo, edad y raza del nombre del archivo
            try:
                parts = filename.split('_')
                print(f"Procesando archivo: {filename}, partes: {parts}")  # Depuración
                if len(parts) < 4:
                    raise ValueError("Formato de nombre de archivo inválido")

                edad = parts[0] if parts[0].isdigit() else "desconocido"
                sexo = sexo_map.get(parts[1], "desconocido") if parts[1] in sexo_map else "desconocido"
                raza = raza_map.get(parts[2], "desconocido") if parts[2] in raza_map else "desconocido"
                print(f"Extraído - Edad: {edad}, Sexo: {sexo}, Raza: {raza}")  # Depuración

            except Exception as parse_error:
                print(f"Error al parsear el nombre del archivo {filename}: {str(parse_error)}")
                edad = "desconocido"
                sexo = "desconocido"
                raza = "desconocido"

            # Preparar datos para el CSV
            csv_row = [
                filename,
                output_path,
                sexo,
                edad,
                raza,
                confidence if confidence is not None else "",
                keypoints['nose'][0] if keypoints and 'nose' in keypoints else "",
                keypoints['nose'][1] if keypoints and 'nose' in keypoints else "",
                keypoints['mouth_right'][0] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['mouth_right'][1] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['right_eye'][0] if keypoints and 'right_eye' in keypoints else "",
                keypoints['right_eye'][1] if keypoints and 'right_eye' in keypoints else "",
                keypoints['left_eye'][0] if keypoints and 'left_eye' in keypoints else "",
                keypoints['left_eye'][1] if keypoints and 'left_eye' in keypoints else "",
                keypoints['mouth_left'][0] if keypoints and 'mouth_left' in keypoints else "",
                keypoints['mouth_left'][1] if keypoints and 'mouth_left' in keypoints else ""
            ]

            # Escribir en el CSV
            with open(csv_path, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(csv_row)

        except Exception as e:
            print(f"Error al procesar {filename}: {str(e)}")
    else:
        print(f"Archivo ignorado (no es imagen): {filename}")

## LOTE 2

In [ ]:
import os
import cv2
import csv
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Dataset', 'dataset-huertas', 'lote-2')
output_folder = os.path.join(shared_drive_path, 'Dataset', 'preprocesadas')
csv_path = os.path.join(shared_drive_path, 'Dataset', 'processed_images.csv')

# Crear carpeta de salida si no existe
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Definir encabezados del CSV
csv_headers = ['filename', 'output_path', 'sexo', 'edad', 'raza', 'mtcnn_confidence',
               'nose_x', 'nose_y', 'mouth_right_x', 'mouth_right_y',
               'right_eye_x', 'right_eye_y', 'left_eye_x', 'left_eye_y',
               'mouth_left_x', 'mouth_left_y']

# Crear CSV si no existe
if not os.path.exists(csv_path):
    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(csv_headers)

# Diccionarios para mapear valores numéricos de sexo y raza
sexo_map = {
    '0': 'Hombre',
    '1': 'Mujer'
}

raza_map = {
    '0': 'Blanco',
    '1': 'Negro',
    '2': 'Asiático',
    '3': 'Indio',
    '4': 'Otros'
}

# Procesar imágenes
for filename in os.listdir(input_folder):
    input_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    # Verificar que el archivo es una imagen
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        try:
            image = cv2.imread(input_path)
            if image is None:
                print(f"Error: No se pudo cargar la imagen {filename}")
                continue
            imagen_procesada, confidence, keypoints = procesar_imagen(image)
            # Convertir de vuelta a formato uint8 para guardar
            imagen_procesada = (imagen_procesada * 255).astype('uint8')
            cv2.imwrite(output_path, imagen_procesada)
            print(f"Imagen procesada: {filename}")

            # Extraer sexo, edad y raza del nombre del archivo
            try:
                parts = filename.split('_')
                print(f"Procesando archivo: {filename}, partes: {parts}")  # Depuración
                if len(parts) < 4:
                    raise ValueError("Formato de nombre de archivo inválido")

                edad = parts[0] if parts[0].isdigit() else "desconocido"
                sexo = sexo_map.get(parts[1], "desconocido") if parts[1] in sexo_map else "desconocido"
                raza = raza_map.get(parts[2], "desconocido") if parts[2] in raza_map else "desconocido"
                print(f"Extraído - Edad: {edad}, Sexo: {sexo}, Raza: {raza}")  # Depuración

            except Exception as parse_error:
                print(f"Error al parsear el nombre del archivo {filename}: {str(parse_error)}")
                edad = "desconocido"
                sexo = "desconocido"
                raza = "desconocido"

            # Preparar datos para el CSV
            csv_row = [
                filename,
                output_path,
                sexo,
                edad,
                raza,
                confidence if confidence is not None else "",
                keypoints['nose'][0] if keypoints and 'nose' in keypoints else "",
                keypoints['nose'][1] if keypoints and 'nose' in keypoints else "",
                keypoints['mouth_right'][0] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['mouth_right'][1] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['right_eye'][0] if keypoints and 'right_eye' in keypoints else "",
                keypoints['right_eye'][1] if keypoints and 'right_eye' in keypoints else "",
                keypoints['left_eye'][0] if keypoints and 'left_eye' in keypoints else "",
                keypoints['left_eye'][1] if keypoints and 'left_eye' in keypoints else "",
                keypoints['mouth_left'][0] if keypoints and 'mouth_left' in keypoints else "",
                keypoints['mouth_left'][1] if keypoints and 'mouth_left' in keypoints else ""
            ]

            # Escribir en el CSV
            with open(csv_path, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(csv_row)

        except Exception as e:
            print(f"Error al procesar {filename}: {str(e)}")
    else:
        print(f"Archivo ignorado (no es imagen): {filename}")

# Lotes Gabriela

## LOTE 1

In [ ]:
import os
import cv2
import csv
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Dataset', 'dataset-gabriela', 'lote-1')
output_folder = os.path.join(shared_drive_path, 'Dataset', 'preprocesadas')
csv_path = os.path.join(shared_drive_path, 'Dataset', 'processed_images.csv')

# Crear carpeta de salida si no existe
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Definir encabezados del CSV
csv_headers = ['filename', 'output_path', 'sexo', 'edad', 'raza', 'mtcnn_confidence',
               'nose_x', 'nose_y', 'mouth_right_x', 'mouth_right_y',
               'right_eye_x', 'right_eye_y', 'left_eye_x', 'left_eye_y',
               'mouth_left_x', 'mouth_left_y']

# Crear CSV si no existe
if not os.path.exists(csv_path):
    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(csv_headers)

# Diccionarios para mapear valores numéricos de sexo y raza
sexo_map = {
    '0': 'Hombre',
    '1': 'Mujer'
}

raza_map = {
    '0': 'Blanco',
    '1': 'Negro',
    '2': 'Asiático',
    '3': 'Indio',
    '4': 'Otros'
}

# Procesar imágenes
for filename in os.listdir(input_folder):
    input_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    # Verificar que el archivo es una imagen
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        try:
            image = cv2.imread(input_path)
            if image is None:
                print(f"Error: No se pudo cargar la imagen {filename}")
                continue
            imagen_procesada, confidence, keypoints = procesar_imagen(image)
            # Convertir de vuelta a formato uint8 para guardar
            imagen_procesada = (imagen_procesada * 255).astype('uint8')
            cv2.imwrite(output_path, imagen_procesada)
            print(f"Imagen procesada: {filename}")

            # Extraer sexo, edad y raza del nombre del archivo
            try:
                parts = filename.split('_')
                print(f"Procesando archivo: {filename}, partes: {parts}")  # Depuración
                if len(parts) < 4:
                    raise ValueError("Formato de nombre de archivo inválido")

                edad = parts[0] if parts[0].isdigit() else "desconocido"
                sexo = sexo_map.get(parts[1], "desconocido") if parts[1] in sexo_map else "desconocido"
                raza = raza_map.get(parts[2], "desconocido") if parts[2] in raza_map else "desconocido"
                print(f"Extraído - Edad: {edad}, Sexo: {sexo}, Raza: {raza}")  # Depuración

            except Exception as parse_error:
                print(f"Error al parsear el nombre del archivo {filename}: {str(parse_error)}")
                edad = "desconocido"
                sexo = "desconocido"
                raza = "desconocido"

            # Preparar datos para el CSV
            csv_row = [
                filename,
                output_path,
                sexo,
                edad,
                raza,
                confidence if confidence is not None else "",
                keypoints['nose'][0] if keypoints and 'nose' in keypoints else "",
                keypoints['nose'][1] if keypoints and 'nose' in keypoints else "",
                keypoints['mouth_right'][0] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['mouth_right'][1] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['right_eye'][0] if keypoints and 'right_eye' in keypoints else "",
                keypoints['right_eye'][1] if keypoints and 'right_eye' in keypoints else "",
                keypoints['left_eye'][0] if keypoints and 'left_eye' in keypoints else "",
                keypoints['left_eye'][1] if keypoints and 'left_eye' in keypoints else "",
                keypoints['mouth_left'][0] if keypoints and 'mouth_left' in keypoints else "",
                keypoints['mouth_left'][1] if keypoints and 'mouth_left' in keypoints else ""
            ]

            # Escribir en el CSV
            with open(csv_path, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(csv_row)

        except Exception as e:
            print(f"Error al procesar {filename}: {str(e)}")
    else:
        print(f"Archivo ignorado (no es imagen): {filename}")

## LOTE 2

In [ ]:
import os
import cv2
import csv
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Rutas
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = os.path.join(shared_drive_path, 'Dataset', 'dataset-gabriela', 'lote-2')
output_folder = os.path.join(shared_drive_path, 'Dataset', 'preprocesadas')
csv_path = os.path.join(shared_drive_path, 'Dataset', 'processed_images.csv')

# Crear carpeta de salida si no existe
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Definir encabezados del CSV
csv_headers = ['filename', 'output_path', 'sexo', 'edad', 'raza', 'mtcnn_confidence',
               'nose_x', 'nose_y', 'mouth_right_x', 'mouth_right_y',
               'right_eye_x', 'right_eye_y', 'left_eye_x', 'left_eye_y',
               'mouth_left_x', 'mouth_left_y']

# Crear CSV si no existe
if not os.path.exists(csv_path):
    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(csv_headers)

# Diccionarios para mapear valores numéricos de sexo y raza
sexo_map = {
    '0': 'Hombre',
    '1': 'Mujer'
}

raza_map = {
    '0': 'Blanco',
    '1': 'Negro',
    '2': 'Asiático',
    '3': 'Indio',
    '4': 'Otros'
}

# Procesar imágenes
for filename in os.listdir(input_folder):
    input_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    # Verificar que el archivo es una imagen
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        try:
            image = cv2.imread(input_path)
            if image is None:
                print(f"Error: No se pudo cargar la imagen {filename}")
                continue
            imagen_procesada, confidence, keypoints = procesar_imagen(image)
            # Convertir de vuelta a formato uint8 para guardar
            imagen_procesada = (imagen_procesada * 255).astype('uint8')
            cv2.imwrite(output_path, imagen_procesada)
            print(f"Imagen procesada: {filename}")

            # Extraer sexo, edad y raza del nombre del archivo
            try:
                parts = filename.split('_')
                print(f"Procesando archivo: {filename}, partes: {parts}")  # Depuración
                if len(parts) < 4:
                    raise ValueError("Formato de nombre de archivo inválido")

                edad = parts[0] if parts[0].isdigit() else "desconocido"
                sexo = sexo_map.get(parts[1], "desconocido") if parts[1] in sexo_map else "desconocido"
                raza = raza_map.get(parts[2], "desconocido") if parts[2] in raza_map else "desconocido"
                print(f"Extraído - Edad: {edad}, Sexo: {sexo}, Raza: {raza}")  # Depuración

            except Exception as parse_error:
                print(f"Error al parsear el nombre del archivo {filename}: {str(parse_error)}")
                edad = "desconocido"
                sexo = "desconocido"
                raza = "desconocido"

            # Preparar datos para el CSV
            csv_row = [
                filename,
                output_path,
                sexo,
                edad,
                raza,
                confidence if confidence is not None else "",
                keypoints['nose'][0] if keypoints and 'nose' in keypoints else "",
                keypoints['nose'][1] if keypoints and 'nose' in keypoints else "",
                keypoints['mouth_right'][0] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['mouth_right'][1] if keypoints and 'mouth_right' in keypoints else "",
                keypoints['right_eye'][0] if keypoints and 'right_eye' in keypoints else "",
                keypoints['right_eye'][1] if keypoints and 'right_eye' in keypoints else "",
                keypoints['left_eye'][0] if keypoints and 'left_eye' in keypoints else "",
                keypoints['left_eye'][1] if keypoints and 'left_eye' in keypoints else "",
                keypoints['mouth_left'][0] if keypoints and 'mouth_left' in keypoints else "",
                keypoints['mouth_left'][1] if keypoints and 'mouth_left' in keypoints else ""
            ]

            # Escribir en el CSV
            with open(csv_path, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(csv_row)

        except Exception as e:
            print(f"Error al procesar {filename}: {str(e)}")
    else:
        print(f"Archivo ignorado (no es imagen): {filename}")